# Conversor de PDFs para Markdown — Painel de Ciências Humanas

Este notebook converte seus PDFs didáticos em arquivos **Markdown** (.md) e gera um **índice** (`indice.json`) — o material que alimenta o repositório do painel de estudos.

**Como usar (não precisa programar):**

1. Clique em **Ambiente de execução > Executar tudo** (ou execute célula por célula, clicando no play à esquerda de cada uma).
2. Escolha **uma** das duas maneiras de entregar os PDFs (Opção A ou Opção B, explicadas abaixo).
3. No final, baixe o arquivo zip gerado e coloque o conteúdo na pasta `acervo/` do repositório.

**Opção A — enviar do computador:** aparece uma janela para escolher os arquivos. Boa para até ~20 PDFs.

**Opção B — ler do Google Drive (recomendada para 100+ PDFs):** o notebook lê a pasta do seu Drive direto, incluindo subpastas por tema.

Se algo der errado, apenas execute a célula novamente. Nada será perdido.

In [ ]:
# CELULA 1 — Preparar as ferramentas (executar sempre)
!pip install -q pymupdf4llm

import pymupdf4llm
print("Ferramentas prontas. Pode seguir para a Opção A ou Opção B.")

## Opção A — Enviar PDFs do computador

Execute a célula abaixo e clique no botão que aparecer para selecionar um ou vários PDFs de uma vez. Aguarde o upload terminar antes de continuar.

*Se você vai usar o Google Drive (Opção B), pule esta célula.*

In [ ]:
# CELULA 2 (OPÇÃO A) — Enviar PDFs do computador
from google.colab import files

enviados = files.upload()  # abre a janela para escolher os arquivos

pdfs_subidos = [nome for nome in enviados if nome.lower().endswith(".pdf")]
print()
print(f"{len(pdfs_subidos)} PDF(s) recebido(s) com sucesso.")

## Opção B — Ler os PDFs do Google Drive (recomendada para 100+)

1. No seu Google Drive, garanta que os PDFs estão na pasta **`Estudos-PDFs`** (pode haver subpastas por tema dentro dela — elas são lidas também).
2. Na célula abaixo, se o nome da sua pasta for outro, **edite apenas o texto entre aspas** na linha `PASTA_DRIVE`.
3. Execute a célula: o Google pedirá permissão para acessar o Drive (é seguro, é sua própria conta).

*Se você já usou a Opção A, pode executar esta também — ou pular.*

In [ ]:
# CELULA 3 (OPÇÃO B) — Ler PDFs do Google Drive
from google.colab import drive
import pathlib

drive.mount('/content/drive')

PASTA_DRIVE = '/content/drive/MyDrive/Estudos-PDFs'  # <-- EDITE AQUI se sua pasta tiver outro nome

if not pathlib.Path(PASTA_DRIVE).exists():
    raise SystemExit(f"A pasta {PASTA_DRIVE} nao foi encontrada. Verifique o nome e execute de novo.")

pdfs_drive = [str(p) for p in sorted(pathlib.Path(PASTA_DRIVE).rglob('*.pdf'))]
print(f"{len(pdfs_drive)} PDF(s) encontrados no Drive:")
for p in pdfs_drive:
    print(" -", p.replace('/content/drive/MyDrive/', ''))

## Conversão — PDFs em Markdown

Esta célula processa **todos** os PDFs entregues (Opção A, Opção B ou as duas). Para cada livro ela mostra o resultado: `OK` ou um aviso de **PDF escaneado** (quando o arquivo é uma imagem das páginas e não tem texto selecionável — a solução está no Guia 02 do repositório).

A conversão de livros grandes pode levar de alguns segundos a alguns minutos por arquivo. É normal.

In [ ]:
# CELULA 4 — Converter todos os PDFs para Markdown
import pathlib, re, unicodedata

# junta o que veio da Opção A e da Opção B (o que não foi usado fica vazio)
lista_pdfs = globals().get('pdfs_subidos', []) + globals().get('pdfs_drive', [])

if not lista_pdfs:
    raise SystemExit("Nenhum PDF entregue. Use a Opção A ou a Opção B antes de executar esta célula.")

def nome_seguro(texto):
    """Remove acentos e espaços para criar nomes de arquivo compatíveis com o GitHub."""
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-zA-Z0-9]+', '-', texto).strip('-').lower()

def tema_do_pdf(caminho):
    """Se o PDF veio do Drive em uma subpasta de tema, usa o nome da subpasta."""
    pasta = pathlib.Path(caminho).parent
    raiz = pasta.name  # nome imediato da pasta que contém o PDF
    if raiz and raiz != 'MyDrive':
        return nome_seguro(raiz)
    return 'sem-tema'

saida = pathlib.Path('/content/markdown')
saida.mkdir(exist_ok=True)

relatorio = []
for caminho in lista_pdfs:
    caminho = pathlib.Path(caminho)
    nome = caminho.stem
    print(f"Convertendo: {nome} ...", end=' ')
    try:
        md_texto = pymupdf4llm.to_markdown(str(caminho))
    except Exception as erro:
        relatorio.append((nome, 'ERRO', 0, 'sem-tema'))
        print(f"ERRO ({erro})")
        continue
    if len(md_texto.strip()) < 300:
        relatorio.append((nome, 'SEM TEXTO (provavelmente escaneado)', 0, tema_do_pdf(caminho)))
        print("sem texto — provavelmente PDF escaneado")
        continue
    destino = saida / (nome_seguro(nome) + '.md')
    destino.write_text(md_texto, encoding='utf-8')
    relatorio.append((nome, 'OK', len(md_texto), tema_do_pdf(caminho)))
    print(f"OK ({len(md_texto):,} caracteres).".replace(',', '.'))

print()
print("=== RESUMO DA CONVERSÃO ===")
for nome, status, tamanho, tema in relatorio:
    print(f"{status:>35} | {tema:<12} | {nome}")

## Geração do índice

Esta célula cria o `indice.json` — o mapa de tudo o que foi convertido. Ele vai para a pasta `indices/` do repositório e alimenta a tabela "Painel de temas" do README.

In [ ]:
# CELULA 5 — Gerar o indice.json
import json, datetime

indice = {
    "projeto": "painel-ciencias-humanas",
    "descricao": "Mapa geral do acervo processado. Gerado automaticamente pelo notebook converter-pdfs.",
    "atualizado_em": datetime.datetime.now().isoformat(timespec='seconds'),
    "livros": []
}

for arquivo_md in sorted(saida.glob('*.md')):
    texto = arquivo_md.read_text(encoding='utf-8')
    registro = {
        "arquivo": arquivo_md.name,
        "titulo": arquivo_md.stem.replace('-', ' ').title(),
        "tema": None,
        "caracteres": len(texto),
        "palavras": len(texto.split()),
    }
    for nome, status, tamanho, tema in relatorio:
        if status == 'OK' and nome_seguro(nome) == arquivo_md.stem:
            registro['tema'] = tema
    indice['livros'].append(registro)

(saida / 'indice.json').write_text(json.dumps(indice, ensure_ascii=False, indent=2), encoding='utf-8')

print(f"indice.json criado com {len(indice['livros'])} livro(s).")
print()
print("Livros no índice:")
for livro in indice['livros']:
    print(f" - {livro['titulo']}  (tema: {livro['tema']} | {livro['palavras']:,} palavras)".replace(',', '.'))

## Baixar o resultado

A última célula empacota tudo em um zip e faz o download começar sozinho. Descompacte no seu computador:

- os arquivos `.md` de livros vão para a pasta **`acervo/`** do repositório;
- o **`indice.json`** vai para a pasta **`indices/`**.

Como subir para o GitHub: veja `docs/04-atualizar-o-github.md` (2 minutos, só arrastar e soltar).

In [ ]:
# CELULA 6 — Empacotar e baixar
import shutil
from google.colab import files as downloader

caminho_zip = shutil.make_archive('/content/acervo-convertido', 'zip', '/content/markdown')
downloader.download(caminho_zip)
print("Download iniciado. Descompacte e envie o conteúdo para o repositório:")
print(" - livros .md  -> pasta acervo/")
print(" - indice.json -> pasta indices/")